# Import Required Libraries

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

from ISLP import load_data
from ISLP.models import summarize

from sklearn.linear_model import LinearRegression


## **Question 8**

This question involves the use of simple linear regression on the Auto
data set.

- Use the `sm.OLS()` function to perform a simple linear regression with mpg as the response and horsepower as the predictor. Use
the `summarize()` function to print the results. Comment on the output. For example:
    - Is there a relationship between the predictor and the response?
    - How strong is the relationship between the predictor and the response?
    - Is the relationship between the predictor and the response positive or negative?
    - What is the predicted mpg associated with a horsepower of 98? What are the associated 95 % confidence and prediction intervals?
- Plot the response and the predictor in a new set of axesax. Use the `ax.axline()` method or the `abline()` function defined in the lab to display the least squares regression line.
- Produce some of diagnostic plots of the least squares regression fit as described in the lab. Comment on any problems you see with the fit. 

In [13]:
auto = load_data("auto")
auto.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


In [19]:
y = auto["mpg"]
X = auto["horsepower"]
X = sm.add_constant(X)
model = sm.OLS(y, X)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.606
Model:                            OLS   Adj. R-squared:                  0.605
Method:                 Least Squares   F-statistic:                     599.7
Date:                Thu, 24 Sep 2026   Prob (F-statistic):           7.03e-81
Time:                        08:48:51   Log-Likelihood:                -1178.7
No. Observations:                 392   AIC:                             2361.
Df Residuals:                     390   BIC:                             2369.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         39.9359      0.717     55.660      0.0

In [20]:
print(summarize(results))

               coef  std err       t  P>|t|
const       39.9359    0.717  55.660    0.0
horsepower  -0.1578    0.006 -24.489    0.0


In [ ]:
data = pd.read_csv("Advertising.csv")
data.drop(columns=["Unnamed: 0"], inplace=True)
data.head()

In [ ]:
Y = data["sales"]
x = data.drop(columns=["sales", "radio", "newspaper"])

x = sm.add_constant(x)

model = sm.OLS(Y, x)
results = model.fit()

results.params

In [ ]:
fStatistics = results.fvalue
rsquared = results.rsquared
rss = np.sqrt(results.scale)

print(f"F-statistics: {fStatistics:.1f}")
print(f"Residual Standard Error: {rss:.3f}")
print(f"R2: {rsquared:.4f}")

In [ ]:
results.summary2()

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 1. Load your dataset (adjust path and column names as needed)
# Example: using Advertising.csv or any tabular data
df = pd.read_csv("Advertising.csv")

# Define target (y) and features (X)
target_col = "sales"
feature_cols = ["TV", "radio", "newspaper"]

y = df[target_col]
X = df[feature_cols]

# 2. Add an intercept column (constant) to the design matrix
X_design = sm.add_constant(X)

# 3. Fit the Ordinary Least Squares (OLS) model
model = sm.OLS(y, X_design)
results = model.fit()

# -------------------------------------------------------------
# Standard Errors of Coefficients
# -------------------------------------------------------------
coef_summary = pd.DataFrame(
    {
        "Coefficient": results.params,
        "Std. Error": results.bse,  # SE of coefficients
        "t-statistic": results.tvalues,
        "p-value": results.pvalues,
    }
)
print("=== Coefficients and Standard Errors ===")
print(coef_summary.round(4))

# -------------------------------------------------------------
# Prediction, Confidence Interval, and Prediction Interval
# -------------------------------------------------------------
# Define new query points for prediction
new_data = pd.DataFrame(
    {
        "TV": [100.0, 200.0, 150.0],
        "radio": [20.0, 40.0, 25.0],
        "newspaper": [15.0, 30.0, 20.0],
    }
)

# Crucial: add constant matching the original model specification
new_X_design = sm.add_constant(new_data, has_constant="add")

# Generate prediction object
pred_results = results.get_prediction(new_X_design)

# summary_frame(alpha=0.05) conveniently returns:
# - mean: Point prediction (y_hat)
# - mean_se: Standard error of the mean prediction
# - mean_ci_lower / mean_ci_upper: 95% Confidence Interval
# - obs_ci_lower / obs_ci_upper:   95% Prediction Interval
interval_df = pred_results.summary_frame(alpha=0.05)

# Formatting the output for clear inspection:
output = pd.concat([new_data, interval_df], axis=1)
print("\n=== Predictions, Confidence Intervals, and Prediction Intervals ===")
print(
    output[
        [
            *feature_cols,
            "mean",
            "mean_ci_lower",
            "mean_ci_upper",
            "obs_ci_lower",
            "obs_ci_upper",
        ]
    ].round(3)
)

# Plot plane of Sales x TV x Radio advertising

In [ ]:
# Load data
df = pd.read_csv('Advertising.csv')

# Fit model for smooth surface
X = df[['TV', 'radio']]
y = df['sales']
model = LinearRegression()
model.fit(X, y)

# Create grid for surface plot
tv_range = np.linspace(df['TV'].min(), df['TV'].max(), 30)
radio_range = np.linspace(df['radio'].min(), df['radio'].max(), 30)
TV_grid, Radio_grid = np.meshgrid(tv_range, radio_range)

# Predict sales on grid
Sales_grid = model.predict(np.c_[TV_grid.ravel(), Radio_grid.ravel()]).reshape(TV_grid.shape)

# Create 3D plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Surface plot of fitted regression model
surf = ax.plot_surface(TV_grid, Radio_grid, Sales_grid, cmap='viridis', alpha=0.6, edgecolor='none')

# Scatter plot of actual data
ax.scatter(df['TV'], df['radio'], df['sales'], color='red', s=30, label='Actual Sales')

ax.set_xlabel('TV Spend ($)')
ax.set_ylabel('Radio Spend ($)')
ax.set_zlabel('Sales')
ax.set_title('Sales as a Function of TV and Radio Advertising')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, label='Predicted Sales')
ax.legend()

plt.show()

# Credit Data set

In [ ]:
credit = load_data("Credit")
credit.head()

In [ ]:
credit.info()

In [ ]:
colunas = ["Balance", 'Age', 'Cards', 'Education', 'Income', 'Limit', 'Rating']
pd.plotting.scatter_matrix(credit[colunas], figsize=(12, 10))
plt.show()

In [ ]:
credit.columns

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))

axes[0, 0].hist(college["Top10perc"], bins=35)
axes[0, 0].set_xlabel("Top10perc")

axes[0, 1].hist(college["PhD"], bins=35)
axes[0, 1].set_xlabel("PhD")

axes[1, 0].hist(college["Accept"], bins=35)
axes[1, 0].set_xlabel("Accept")

axes[1, 1].hist(college["Enroll"], bins=35)
axes[1, 1].set_xlabel("Enroll")

plt.suptitle("Histograms of College Data")
plt.show()

# **Working with the Auto dataset**

In [ ]:
auto = load_data('Auto')
auto.dropna(inplace=True)
auto.head()

In [ ]:
for x in ['cylinders', 'year', 'origin']:
    print(f"{x}: {auto[x].unique()}", "\n")

In [ ]:
for x in auto.columns:
    print(f"Max of {x}: {auto[x].max():.2f}")
    print(f"Min of {x}: {auto[x].min():.2f}")
    print(f"Mean of {x}: {auto[x].mean():.2f}")
    print(f"Std of {x}: {auto[x].std():.2f}\n")

In [ ]:
mask = [(i < 10) or (i > 84) for i in range(len(auto))]
auto_filtered = auto.iloc[mask]

for x in auto_filtered.columns:
    print(f"Max of {x}: {auto_filtered[x].max():.2f}")
    print(f"Min of {x}: {auto_filtered[x].min():.2f}")
    print(f"Mean of {x}: {auto_filtered[x].mean():.2f}")
    print(f"Std of {x}: {auto_filtered[x].std():.2f}\n")



In [ ]:
pd.plotting.scatter_matrix(auto, figsize=(10, 10));

# **Working with the Boston dataset**

In [ ]:
boston = load_data('Boston')
boston.head()

In [ ]:
boston.info()

In [ ]:
boston["chas"].sum()

What is the median pupil-teacher ratio among the towns in this
data set?

In [ ]:
resp = boston["ptratio"].median()
print(f"The median pupil-teacher ratio in the Boston dataset is: {resp:.2f}")

Which suburb of Boston has lowest median value of owner-occupied homes? What are the values of the other predictors for that suburb, and how do those values compare to the overall ranges for those predictors? Comment on your findings.

In [ ]:
boston["age"].median()

In [ ]:
import scipy.stats as stats

# Given parameters
beta1_hat = 0.5
se_beta1 = 0.2
alpha = 0.05

# Critical Z-value for a two-tailed test at alpha = 0.05
z_crit = stats.norm.ppf(1 - alpha / 2)

# Calculate upper and lower bounds for b (95% CI)
b_max = beta1_hat + z_crit * se_beta1
b_min = beta1_hat - z_crit * se_beta1

print(f"Critical Z-value: {z_crit:.4f}")
print(f"Non-rejection region for b: [{b_min:.4f}, {b_max:.4f}]")
print(f"Largest value of b (2 significant digits): {b_max:.2g}")

In [ ]:
stats.norm.sf

In [ ]:
def logReg(beta0, beta1, beta2, x1, x2):
    eq = beta0 + beta1 * x1 + beta2 * x2
    return np.exp(eq) / (1 + np.exp(eq))

In [ ]:
x = logReg(-6, 0.05, 1, 40, 3.5)
print(f"Prob = {x:.2f}")

In [ ]:
beta0 = -6
beta1 = 0.05
beta2 = 1
x2 = 3.5
x1 = (-beta0 - beta2 * x2) / beta1
print(x1)